Retreive 20 records including the last 4 columns from the GCS's file generated from ex01.ipynb
Using "gemini-3.6-flash" (or a version it recommends), ask it to summarize the reports.

- Pre-requisite : Generate API keys through [Google AI Studio](https://aistudio.google.com/app/apikey)

In [1]:
import datetime
import json
import os

from dotenv import load_dotenv
from google import genai
from google.oauth2 import service_account
from google.cloud import storage

In [2]:
load_dotenv()

True

In [3]:
gemini_api_key = os.getenv("GEMINI_API_KEY")
service_account_key = os.getenv("GCP_SERVICE_ACCOUNT_KEY")
project_id = os.getenv("GCP_PROJECT_ID")
bucket_name = os.getenv("GCP_BUCKET_NAME")
file_name = f"sf_police_report/{datetime.date.today()}.json"

In [4]:
def retrieve_data_from_gcs(service_account_key: str,
                           project_id: str,
                           bucket_name: str,
                           file_name: str,
                           key_list: list
                           ) -> list:
    credentials = service_account.Credentials.from_service_account_file(service_account_key)
    client = storage.Client(project=project_id,
                            credentials=credentials)
    bucket = client.bucket(bucket_name)
    file = bucket.blob(file_name)
    content = json.loads(file.download_as_string())

    output = []
    for data in content:
        vals = []

        for key in key_list:
            vals.append(data.get(key, None))
        output.append(vals)
    return output

In [5]:
key_list = ['incident_datetime', 'report_datetime', 'incident_code',
            'incident_category', 'incident_description', 'latitude',
            'longitude', 'police_district']
data = retrieve_data_from_gcs(service_account_key, 
                              project_id,
                              bucket_name,
                              file_name,
                              key_list)

In [6]:
filtered_data = [row[-4:] for row in data][:20]

In [7]:
# The client gets the API key from the environment variable `GEMINI_API_KEY`.
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [8]:
model_name = "gemini-3.6-flash"

In [9]:
prompt_content = f"There has been a police report\
on the following list of [description, lon, lat, district] : {filtered_data} recently.\
Summarize the reports"

In [10]:
response = client.models.generate_content(model=model_name,
                                          contents=prompt_content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [11]:
response.text

'Here is a summary of the 20 police reports provided, broken down by crime category, district, and key observations:\n\n---\n\n### **Executive Summary**\nA total of **20 incidents** were reported across **7 San Francisco police districts**, as well as **2 incidents originating out of SF**. The majority of reports involve **property crimes/thefts (30%)** and **violent offenses/crimes against persons (25%)**.\n\n---\n\n### **Breakdown by Incident Category**\n\n1. **Theft & Property Crimes (6 reports)**\n   * **Theft from Unlocked Vehicles (>$950):** 2 incidents (*Ingleside, Mission*)\n   * **Theft from Building (>$950):** 1 incident (*Richmond*)\n   * **Petty Theft ($50–$200):** 1 incident (*Mission*)\n   * **Residential Burglary (Unlawful Entry):** 1 incident (*Northern*)\n   * **Stolen Vehicle:** 1 incident (*Ingleside*)\n\n2. **Violent Crimes & Assaults (5 reports)**\n   * **Robbery with Force:** 2 incidents (*Mission, Northern*)\n   * **Battery with Serious Injuries:** 1 incident (*C

In [12]:
# Note: This is an optional to display markdown strings.
from rich.console import Console
from rich.markdown import Markdown

console = Console()
md = Markdown(response.text)
console.print(md)


Here is a summary of the 20 police reports provided, broken down by crime category, district, and key observations:

-------------------------------------------------------------------------------------------------------------------

Executive Summary                                                                                                  

A total of 20 incidents were reported across 7 San Francisco police districts, as well as 2 incidents originating  
out of SF. The majority of reports involve property crimes/thefts (30%) and violent offenses/crimes against persons
(25%).                                                                                                             

-------------------------------------------------------------------------------------------------------------------

Breakdown by Incident Category                                                                                     

 1 Theft & Property Crimes (6 reports)                                                                             
    • Theft from Unlocked Vehicles (>$950): 2 incidents (Ingleside, Mission)                                       
    • Theft from Building (>$950): 1 incident (Richmond)                                                           
    • Petty Theft ($50–$200): 1 incident (Mission)                                                                 
    • Residential Burglary (Unlawful Entry): 1 incident (Northern)                                                 
    • Stolen Vehicle: 1 incident (Ingleside)                                                                       
 2 Violent Crimes & Assaults (5 reports)                                                                           
    • Robbery with Force: 2 incidents (Mission, Northern)                                                          
    • Battery with Serious Injuries: 1 incident (Central)                                                          
    • Aggravated Assault with Force: 1 incident (Bayview)                                                          
    • Simple Battery: 1 incident (Ingleside)                                                                       
 3 Drug Offenses (2 reports)                                                                                       
    • Heroin Offense: 1 incident (Tenderloin)                                                                      
    • Possession of Narcotics Paraphernalia: 1 incident (Mission)                                                  
 4 Other Offenses & Investigations (7 reports)                                                                     
    • Restraining Order Violation: 1 incident (Ingleside)                                                          
    • False Personation / Impersonation: 1 incident (Bayview)                                                      
    • Municipal Code Violation: 1 incident (Tenderloin)                                                            
    • Suspicious Occurrence: 1 incident (Richmond)                                                                 
    • Miscellaneous Investigation: 1 incident (Ingleside)                                                          
    • Recovered Vehicle: 1 incident (Out of SF)                                                                    
    • Warrant Arrest (SF Warrant): 1 incident (Out of SF)                                                          

-------------------------------------------------------------------------------------------------------------------

Breakdown by District                                                                                              

 • Ingleside (5 reports): Highest volume of reports. Included battery, stolen vehicle, theft from an unlocked      
   vehicle, violation of a restraining order, and a general investigation.                                         
 • Mission (4 reports): Included drug paraphern